In [ ]:
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import TargetEncoder
from dateutil.relativedelta import relativedelta
from ngboost import NGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
seed = 42
y_col_name = "Aotizhongxin_PM2.5"

## 1. データの読み込み

In [ ]:
df_data = pd.read_csv("./data/prepro_data.csv", encoding="utf-8")

In [ ]:
df_data.head(3)

In [ ]:
df_data.isnull().sum()

In [ ]:
# 欠損が全体に占める割合
df_nan_ratio = df_data.isnull().sum().to_frame().T/len(df_data)

# 欠損の割合が30%以上のカラムを確認→該当なし。カラムは捨てずに全部特徴量として使う。
print("over 30%", df_nan_ratio.loc[0][df_nan_ratio.loc[0] >= 0.3].index.tolist())

print("over 10%", df_nan_ratio.loc[0][df_nan_ratio.loc[0] >= 0.1].index.tolist())


In [ ]:
df_data[y_col_name].isnull().sum()

# 目的変数に欠損があるが、時系列データなので、欠損行をdropせずに、今回は他の数値データと同様に前後の値を参考にした線形補完で代用

## 2. 前処理

In [ ]:
df_prepro = df_data.copy()

### 2.1 数値データの線形補完

In [ ]:
# 欠損値補完
# 前後の値を参照する
num_cols = df_prepro.select_dtypes(include="number").columns
df_prepro[num_cols] = df_prepro[num_cols].interpolate(method="linear") 

# Gucheng_NO2のnanが連続している行は補完できない。

### 2.2 カテゴリ型データの補完

In [ ]:
obj_cols = df_prepro.select_dtypes(exclude="number").columns
obj_cols = obj_cols.drop("datetime")

for col in obj_cols:

    mode_value = df_prepro[col].mode()[0] #複数あったときのために、[0]

    df_prepro[col] = df_prepro[col].fillna(mode_value)
    df_prepro[col] = df_prepro[col].astype("category")

In [ ]:
df_prepro.dtypes

In [ ]:
df_prepro.select_dtypes(exclude="number").columns

In [ ]:
df_prepro["Guanyuan_wd"].value_counts()

### 2.3 datetimeの前処理

In [ ]:
df_prepro["datetime"].isnull().sum()

In [ ]:
df_prepro["datetime"] = pd.to_datetime(df_prepro["datetime"])

In [ ]:
df_prepro["year"] = df_prepro["datetime"].dt.year
df_prepro["month"] = df_prepro["datetime"].dt.month
df_prepro["day_of_year"] = df_prepro["datetime"].dt.dayofyear
df_prepro["hour"] = df_prepro["datetime"].dt.hour

In [ ]:
# sin/cos化

df_prepro["month_sin"] = np.sin(2 * np.pi * df_prepro["month"] / 12)
df_prepro["month_cos"] = np.cos(2 * np.pi * df_prepro["month"] / 12)

max_days_year = np.where(df_prepro["datetime"].dt.is_leap_year, 366, 365)
df_prepro["doy_sin"] = np.sin(2 * np.pi * df_prepro["day_of_year"] / max_days_year)
df_prepro["doy_cos"] = np.cos(2 * np.pi * df_prepro["day_of_year"] / max_days_year)

df_prepro["hour_sin"] = np.sin(2 * np.pi * df_prepro["hour"] / 24)
df_prepro["hour_cos"] = np.cos(2 * np.pi * df_prepro["hour"] / 24)



In [ ]:
df_prepro.head(3)

### 2.4　 dfの先頭から連続している欠損値処理

In [ ]:
df_prepro[df_prepro.isna().any(axis=1)]

In [ ]:
df_prepro.shape

In [ ]:
# Gucheng_NO2で先頭の２０行が連続してnanである。
# 先頭で連続しているため、.interpolate(method="linear") で補完できない。

df_prepro = df_prepro.bfill() # 20行目の値で、0~19行目を埋める

# df_prepro = df_prepro.dropna() # 先頭２０行だけなので、全体に対して、20/35064＝0.05％であり、そもそも削除する方法もあり

### 2.5 目的変数の時刻をシフト

In [ ]:
# 目的変数を3時間シフトし、3時間先予測タスクを作成

In [ ]:
# # 目的地点以外のPM2.5のカラムを削除
# 他地点PM2.5は目的変数と同一物理量であり、現実の予測環境では利用できないケースを想定して除外した

y = df_prepro[y_col_name]

drop_col = df_prepro.filter(like="PM2.5").columns
df_prepro = df_prepro.drop(drop_col, axis=1)
df_prepro[y_col_name] = y

In [ ]:
# 現在時刻tの特徴量から、3時間後(t+3)のPM2.5を予測する
df_prepro[y_col_name] = df_prepro[y_col_name].shift(-3)

In [ ]:
# 目的変数の3時間シフトを作成したために発生したnanを削除する
df_prepro = df_prepro.dropna()

In [ ]:
df_prepro.head(3)

### 2.6　相関が１の特徴量を片方削除

In [ ]:
# edaで調べたものを指定

corr1_drop_cols = [
    "Tiantan_TEMP", "Tiantan_PRES", "Tiantan_DEWP", "Tiantan_RAIN", "Tiantan_WSPM",
    "Changping_DEWP", "Changping_RAIN", "Changping_WSPM",
    "Guanyuan_TEMP", "Guanyuan_DEWP", "Guanyuan_RAIN", "Guanyuan_WSPM"
]

# edaの時の結果↓
# Tiantan_TEMP    Dongsi_TEMP          1.0
# Tiantan_PRES    Dongsi_PRES          1.0
# Tiantan_DEWP    Dongsi_DEWP          1.0
# Tiantan_RAIN    Dongsi_RAIN          1.0
# Tiantan_WSPM    Dongsi_WSPM          1.0
# Changping_DEWP  Dingling_DEWP        1.0
# Changping_RAIN  Dingling_RAIN        1.0
# Changping_WSPM  Dingling_WSPM        1.0
# Guanyuan_TEMP   Aotizhongxin_TEMP    1.0
# Guanyuan_DEWP   Aotizhongxin_DEWP    1.0
# Guanyuan_RAIN   Aotizhongxin_RAIN    1.0
# Guanyuan_WSPM   Aotizhongxin_WSPM    1.0

In [ ]:
df_prepro = df_prepro.drop(corr1_drop_cols, axis=1)

## 3. データセット作成

In [ ]:
# データセットの初日と最終日の確認
print(f"{df_prepro["datetime"].min()}\n{df_prepro["datetime"].max()}")

In [ ]:
# train:val:test=2:1:1にわける

train_start_date = df_prepro["datetime"].min()
val_start_date = train_start_date + relativedelta(years=2) #　閏年を考慮して2年をたす
test_start_date = val_start_date + relativedelta(years=1)

In [ ]:
print(train_start_date, val_start_date, test_start_date)

In [ ]:

df_train = df_prepro[df_prepro["datetime"] < val_start_date].set_index("datetime")
df_val = df_prepro[(df_prepro["datetime"] >= val_start_date) & (df_prepro["datetime"] < test_start_date)].set_index("datetime")
df_test = df_prepro[df_prepro["datetime"] >= test_start_date].set_index("datetime")

# drop ver
# df_train = df_prepro[df_prepro["datetime"] < val_start_date].drop("datetime", axis=1)
# df_val = df_prepro[(df_prepro["datetime"] >= val_start_date) & (df_prepro["datetime"] < test_start_date)].drop("datetime", axis=1)
# df_test = df_prepro[df_prepro["datetime"] >= test_start_date].drop("datetime", axis=1)


X_train = df_train.drop(y_col_name, axis=1)
y_train =df_train[y_col_name]
y_train.name = y_col_name

X_val = df_val.drop(y_col_name, axis=1)
y_val = df_val[y_col_name]
y_val.name = y_col_name

X_test = df_test.drop(y_col_name, axis=1)
y_test = df_test[y_col_name]
y_test.name = y_col_name

In [ ]:
# カテゴリ型を数値に変換
# onehot encodingだと次元が多く増えるので、targetencoderで目的変数の値に応じて、カテゴリを数値化する
cat_to_num_cols = X_train.select_dtypes(exclude="number").columns

encoder = TargetEncoder(cols=cat_to_num_cols, handle_missing="value") # 未知のカテゴリは平均値で埋める

X_train_encoded = encoder.fit_transform(X_train, y_train)
X_val_encoded = encoder.transform(X_val)
X_test_encoded = encoder.transform(X_test)

## 4. モデルの学習と評価

In [ ]:
ngb = NGBRegressor(random_state=seed)

In [ ]:
ngb.fit(
    X_train_encoded, y_train,
    X_val=X_val_encoded, 
    Y_val=y_val, 
    early_stopping_rounds=10
)

In [ ]:
y_pred = ngb.predict(X_test_encoded)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [ ]:
# 精度の確認
print(f"MAE:{mae}\nRMSE:{rmse}")

In [ ]:
# 精度の目安のために、目的変数のテストデータの分布の確認
y_test.describe()

In [ ]:
# ngboostの特徴である、確率分布を計算
Y_dist = ngb.pred_dist(X_test_encoded)

In [ ]:
len(y_pred)

In [ ]:
X_test_encoded.head(3)

In [ ]:
X_test_encoded.index

In [ ]:
# 予測値の表示

sns.set_theme(style="whitegrid")

datetime_start =  datetime(2016, 4, 1, 0, 0) + timedelta(hours=3) #　予測対象は3時間後なので表示時刻を+3時間補正
datetime_end = datetime(2016, 4, 3, 0, 0) + timedelta(hours=3)
time_slicer = (X_test_encoded.index >= datetime_start) & (X_test_encoded.index <= datetime_end)

# 予測値と分散を取得
y_preds_all = Y_dist.loc
scale_all = Y_dist.scale
y_test_all = y_test.values if hasattr(y_test, "values") else np.array(y_test)

# 表示区間のデータの切り出し
y_preds = y_preds_all[time_slicer]
scale = scale_all[time_slicer]
y_test_slice = y_test_all[time_slicer]

# ９５％信頼区間の上限下限を計算
lower_bound = y_preds - 1.96 * scale
upper_bound = y_preds + 1.96 * scale

# lower_bound, upper_bound = Y_dist.interval(0.95)

# グラフの描画
plt.figure(figsize=(12, 6))

x_axis = X_test_encoded.index[time_slicer]

# 実績値をプロット（見やすくするため点と線の両方を表示）
plt.plot(x_axis, y_test_slice, label="Actual", color="black", alpha=0.6, linestyle="--", marker="o")

# 予測値（平均）をプロット
plt.plot(x_axis, y_preds, label="Predicted", color="#1f77b4", lw=2, marker="s")

# 95%区間の上限・下限の線をプロット
plt.plot(x_axis, upper_bound, color="#1f77b4", alpha=0.3, lw=1)
plt.plot(x_axis, lower_bound, color="#1f77b4", alpha=0.3, lw=1)

#  間のエリアを色付け
plt.fill_between(x_axis, lower_bound, upper_bound, 
                 color="#1f77b4", alpha=0.15, label="95% Confidence Interval")

plt.title(f"NGBoost Prediction (From {datetime_start} to {datetime_end})", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel(f"{y_col_name} Value", fontsize=12)
plt.xticks(x_axis) 
plt.legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.show()